# 09｜五因子模型：品質與保守投資的超額報酬

**學術來源：** Fama & French (2015), *A Five-Factor Asset Pricing Model*, Journal of Financial Economics

Fama-French 在三因子模型（市場、規模、價值）加入兩個新因子：
- **RMW（Robust Minus Weak）**：高獲利能力公司 vs 低獲利能力公司
- **CMA（Conservative Minus Aggressive）**：保守投資公司 vs 激進投資公司

這兩個因子能解釋更多股票報酬的截面差異，也給「選好公司」提供了學術背書。

## 「買好公司」這句話，有什麼問題？

幾乎所有投資書籍都說：找有護城河的優質公司、長期持有。
巴菲特、查理蒙格也是這樣說的。

但沒有人告訴你：「好公司」怎麼定義？

高市占率？高毛利率？高 ROE？配息穩定？管理層誠信？

這些都對，但都很主觀，也很難量化到可以系統性執行的程度。

2015 年，Fama 和 French 在三因子的基礎上加了兩個因子——把「好公司」量化：

- **RMW（Robust Minus Weak）**：獲利能力強的公司 vs 獲利能力弱的公司
- **CMA（Conservative Minus Aggressive）**：資本支出保守 vs 激進擴張的公司

結果：高盈利能力 + 保守投資的公司，確實有系統性的超額報酬。

但「品質溢酬」有多大？在壞市場還存在嗎？這一章用數據回答。

## 🎯 學習目標
1. 理解 RMW（品質/獲利能力）與 CMA（保守投資）兩個新增因子
2. 計算五個因子的夏普比率並比較相對強弱
3. 學會用「品質+價值」組合提升長期風險調整後報酬

## 1｜三因子到五因子的演進

| 因子 | 意義 | 代理變數 |
|------|------|----------|
| Mkt-RF | 市場風險溢酬 | 市場報酬 − 無風險利率 |
| SMB | 規模效應（小公司溢酬） | 小市值 − 大市值 |
| HML | 價值效應（高BM溢酬） | 高Book/Market − 低Book/Market |
| **RMW** | **獲利能力（品質）溢酬** | **高ROE − 低ROE** |
| **CMA** | **保守投資溢酬** | **低資本支出成長 − 高資本支出成長** |

五因子模型的模型式：
$$R_i - R_f = \alpha + b_1(Mkt-RF) + b_2 SMB + b_3 HML + b_4 RMW + b_5 CMA + \varepsilon$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = [
    'Microsoft YaHei', 'SimHei', 'Heiti TC',
    'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif'
]
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import pathlib, os

DATA_DIR = pathlib.Path('data')
DATA_DIR.mkdir(exist_ok=True)

FF5_CSV = DATA_DIR / 'ff5_factors.csv'

if FF5_CSV.exists():
    df = pd.read_csv(FF5_CSV, index_col=0, parse_dates=True)
    print(f'從快取載入：{len(df)} 筆月度資料')
else:
    import pandas_datareader.data as web
    raw = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start='1963-07')[0]
    df = raw / 100
    df.index = pd.to_datetime(df.index.to_timestamp())
    df.to_csv(FF5_CSV)
    print(f'下載完成：{len(df)} 筆月度資料')

df.head()

## 2｜五個因子的年化報酬與夏普比率比較

In [ ]:
factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
factor_labels = {
    'Mkt-RF': '市場溢酬\n(Mkt-RF)',
    'SMB':    '規模效應\n(SMB)',
    'HML':    '價值效應\n(HML)',
    'RMW':    '品質效應\n(RMW)',
    'CMA':    '保守投資\n(CMA)'
}

ann_ret  = df[factors].mean() * 12
ann_vol  = df[factors].std() * np.sqrt(12)
sharpe   = ann_ret / ann_vol

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#9C27B0', '#FF9800', '#4CAF50', '#F44336']
labels = [factor_labels[f] for f in factors]

bars1 = axes[0].bar(labels, ann_ret * 100, color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('五因子年化報酬 (%)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('年化報酬 (%)')
for bar, val in zip(bars1, ann_ret * 100):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.3 if val >= 0 else val - 0.5,
                 f'{val:.1f}%', ha='center', va='bottom' if val >= 0 else 'top', fontsize=10)

bars2 = axes[1].bar(labels, sharpe, color=colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('五因子夏普比率', fontsize=14, fontweight='bold')
axes[1].set_ylabel('夏普比率')
for bar, val in zip(bars2, sharpe):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01 if val >= 0 else val - 0.02,
                 f'{val:.2f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=10)

plt.tight_layout()
plt.savefig('data/ff5_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n各因子統計：')
print(pd.DataFrame({'年化報酬(%)': ann_ret*100, '年化波動(%)': ann_vol*100, '夏普比率': sharpe}).round(3))

## 3｜RMW（品質因子）深度分析

**RMW** 代表「高獲利 minus 弱獲利」——買入高 ROE/毛利率公司、放空低獲利公司的報酬。

這個因子在學術上有堅實基礎：
- **Novy-Marx (2013)**：毛利率是最強的獲利能力代理指標
- **Fama & French (2015)**：RMW 在五因子中有最高的統計顯著性
- 經濟直覺：優質公司有持續競爭優勢，長期股東受益

In [ ]:
# RMW 滾動10年夏普比率 → 檢驗是否跨時期穩定
window = 120  # 10年 = 120個月
rmw = df['RMW']

rolling_mean  = rmw.rolling(window).mean() * 12
rolling_vol   = rmw.rolling(window).std() * np.sqrt(12)
rolling_sharpe = rolling_mean / rolling_vol

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 累積報酬
cum = (1 + rmw).cumprod()
axes[0].plot(cum.index, cum, color='#4CAF50', linewidth=1.5)
axes[0].fill_between(cum.index, 1, cum, where=(cum >= 1), alpha=0.2, color='#4CAF50')
axes[0].fill_between(cum.index, 1, cum, where=(cum < 1),  alpha=0.2, color='#F44336')
axes[0].axhline(1, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('RMW（品質因子）累積報酬（1963–至今）', fontsize=13, fontweight='bold')
axes[0].set_ylabel('累積倍數')
axes[0].set_yscale('log')

# 滾動夏普
axes[1].plot(rolling_sharpe.index, rolling_sharpe, color='#2196F3', linewidth=1.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(rolling_sharpe.mean(), color='orange', linewidth=1.2, linestyle='--', label=f'平均={rolling_sharpe.mean():.2f}')
axes[1].fill_between(rolling_sharpe.index, 0, rolling_sharpe,
                     where=(rolling_sharpe >= 0), alpha=0.2, color='#2196F3')
axes[1].fill_between(rolling_sharpe.index, 0, rolling_sharpe,
                     where=(rolling_sharpe < 0),  alpha=0.2, color='#F44336')
axes[1].set_title('RMW 滾動10年夏普比率', fontsize=13, fontweight='bold')
axes[1].set_ylabel('夏普比率')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/rmw_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

pct_positive = (rolling_sharpe.dropna() > 0).mean()
print(f'RMW 夏普比率為正的月份占比：{pct_positive:.1%}')
print(f'RMW 整體夏普比率：{(rmw.mean()*12)/(rmw.std()*np.sqrt(12)):.3f}')

## 4｜CMA（保守投資因子）與品質價值組合

**CMA** 捕捉「少投資」的公司溢酬：
- **理論**：Modigliani-Miller 的投資規則——NPV>0 才投資，否則應還給股東
- **實證**：大量資本支出往往預示未來報酬下降（過度自信的管理層）
- **Titman et al. (2004)**：高 capex 增長公司未來三年股票報酬顯著偏低

**RMW + HML（品質+價值）** 組合是許多價值投資者的最愛。

In [ ]:
# 品質+價值組合 vs 單因子
combo = (df['RMW'] + df['HML']) / 2

series_dict = {
    'Mkt-RF（市場）': df['Mkt-RF'],
    'HML（價值）':    df['HML'],
    'RMW（品質）':    df['RMW'],
    '品質+價值 組合': combo,
}

fig, ax = plt.subplots(figsize=(13, 6))
style_map = {'Mkt-RF（市場）': ('#9E9E9E', 0.7, 1.2),
             'HML（價值）':    ('#FF9800', 0.85, 1.5),
             'RMW（品質）':    ('#4CAF50', 0.85, 1.5),
             '品質+價值 組合': ('#E91E63', 1.0, 2.5)}

for name, s in series_dict.items():
    cum = (1 + s).cumprod()
    color, alpha, lw = style_map[name]
    ax.plot(cum.index, cum, label=name, color=color, alpha=alpha, linewidth=lw)

ax.set_yscale('log')
ax.set_title('品質+價值組合 vs 單因子累積報酬（對數尺度）', fontsize=14, fontweight='bold')
ax.set_ylabel('累積倍數（對數）')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('data/quality_value_combo.png', dpi=150, bbox_inches='tight')
plt.show()

for name, s in series_dict.items():
    ann_r = s.mean() * 12
    ann_v = s.std() * np.sqrt(12)
    sh = ann_r / ann_v
    print(f'{name:15s}: 年化報酬={ann_r*100:.2f}%，夏普={sh:.3f}')

## 5｜這跟你有什麼關係？

### 五因子告訴你的三件事

**① 選「賺錢的公司」有統計基礎**

RMW 因子的存在告訴你：長期持有高 ROE、高毛利率公司，確實有超額報酬。
這不是股市玄學，而是 60 年美股數據的實證結果。

**② 「不亂花錢的公司」也有超額報酬**

CMA 因子：管理層把錢存起來或還給股東（配息/回購），比一直擴張的公司報酬更好。
檢驗方式：看「自由現金流/資本支出比率」。FCF > capex 的公司才是保守投資者。

**③ 品質+價值 > 單一因子**

RMW（品質）和 HML（低估值）組合的夏普比率，比任何單一因子都高。
實務上的篩選條件：**ROE > 15% 且 P/B < 3 且 FCF > 0**

### 台灣投資人的操作方向

| 方法 | 說明 |
|------|------|
| **買因子 ETF** | 美國有 QUAL（品質）、MOAT（護城河）等 ETF 直接捕捉 RMW |
| **個股篩選** | 用 CMoney/Goodinfo 篩 ROE>15%、負債比<40%、連續3年正FCF |
| **避開陷阱** | 大幅增資、頻繁購廠的公司→ 小心 CMA 反轉 |

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| RMW（品質） | 高 ROE/毛利率公司長期有超額報酬 |
| CMA（保守） | 少資本支出公司優於大量擴張公司 |
| RMW + HML | 品質+價值組合夏普比率高於任何單一因子 |
| 個人應用 | 篩選條件：ROE > 15%、P/B < 3、自由現金流 > 0 |

> **下一章：** 動能因子——為什麼強者恆強？